In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
players = pd.read_csv("../data/processed/players_master.csv")
players.head()
columns = players.columns
france = players[players["nationality_mapped"]=="France"]
print(france)

KeyError: 'nationality_mapped'

using player overall as the weight for computing averages

In [ ]:
def weighted_avg(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

In [ ]:
france = players[players['team'] == 'France']
france_att = france[france['broad_pos'] == 'ATT']

atk_overall   = weighted_avg(france_att, 'overall', 'overall')
atk_shooting  = weighted_avg(france_att, 'shooting', 'overall')
atk_pace      = weighted_avg(france_att, 'pace', 'overall')
atk_dribbling = weighted_avg(france_att, 'dribbling', 'overall')
print(atk_overall)
print(france_att)

82.50522648083624
       team  no position                  name         dob  caps  goals  \
838  France   7       FW       Ousmane Dembélé  1997-05-15    59      7   
840  France   9       FW         Marcus Thuram  1997-08-06    34      3   
841  France  10       FW         Kylian Mbappé  1998-12-20    98     56   
842  France  11       FW         Michael Olise  2001-12-12    17      7   
843  France  12       FW       Bradley Barcola  2002-09-02    20      3   
851  France  20       FW           Désiré Doué  2005-06-03     7      2   
853  France  22       FW  Jean-Philippe Mateta  1997-06-28     4      2   

                    club nationality_mapped             name_norm  ...  pace  \
838  Paris Saint-Germain             France       ousmane dembele  ...  67.0   
840          Inter Milan             France         marcus thuram  ...  80.0   
841          Real Madrid             France         kylian mbappe  ...  97.0   
842        Bayern Munich             France         michael o

Low depth score = bench is close in quality to starters = deep squad
High depth score = massive dropoff when starters come off = shallow squad

sqaud depth - difference between avg overall bw squad and bench - closer is better

In [ ]:
import unicodedata

def norm(s):
    s = str(s).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

# load and filter understat
understat = pd.read_csv("../data/raw/understat_player_stats.csv")
understat = understat[understat['league'] != 'RUS-Premier League']
understat = understat[understat['time'] >= 200]

# manual corrections for confirmed fuzzy matches
corrections = {
    'lee jae-sung'        : 'jae-sung lee',
    'hwang hee-chan'      : 'hee-chan hwang',
    'azzedine ounahi'     : 'azz-eddine ounahi',
    'andy robertson'      : 'andrew robertson',
    'altay bayındır'      : 'altay bayindir',
    'ferdi kadıoğlu'      : 'ferdi kadioglu',
    'willian pacho'       : 'william pacho',
    'yeremy pino'         : 'yeremi pino',
    'ali yousif'          : 'ali youssif',
    'martin ødegaard'     : 'martin odegaard',
    'hicham boudaoui'     : 'hichem boudaoui',
    'rayan aït-nouri'     : 'rayan ait nouri',
    'mohamed amoura'      : 'mohammed amoura',
    'phillipp mwene'      : 'philipp mwene',
    'abdukodir khusanov'  : 'abduqodir khusanov',
    'luiz henrique'       : 'luis henrique',
}

# norm both sides
understat['name_norm'] = understat['player_name'].apply(norm)
players['name_norm2'] = players['name'].apply(norm)

# apply corrections to players side
players['name_lookup'] = players['name_norm2'].replace(corrections)

# compute xG/90 and xA/90 on understat side
understat['xG_per90'] = understat['xG'] / (understat['time'] / 90)
understat['xA_per90'] = understat['xA'] / (understat['time'] / 90)

# build lookup dict: norm_name → (xG_per90, xA_per90, minutes)
understat = understat.sort_values('time', ascending=False).drop_duplicates(subset='name_norm', keep='first')

und_lookup = understat.set_index('name_norm')[['xG_per90', 'xA_per90', 'time']].to_dict('index')


In [ ]:
rows = []
for team_name in players['team'].unique():
    team = players[players['team']== team_name]
    ATT = team[team['broad_pos']=='ATT']
    
    # match ATT players to understat
    att_und_rows = []
    for _, p in ATT.iterrows():
        lookup_name = p['name_lookup']
        if lookup_name in und_lookup:
            att_und_rows.append({
                'xG_per90' : und_lookup[lookup_name]['xG_per90'],
                'xA_per90' : und_lookup[lookup_name]['xA_per90'],
                'minutes'  : und_lookup[lookup_name]['time']
            })

    if att_und_rows:
        att_und = pd.DataFrame(att_und_rows)
        atk_xG = (att_und['xG_per90'] * att_und['minutes']).sum() / att_und['minutes'].sum()
        atk_xA = (att_und['xA_per90'] * att_und['minutes']).sum() / att_und['minutes'].sum()
    else:
        atk_xG = None
        atk_xA = None

    MID = team[team['broad_pos']=='MID']
    DEF = team[team['broad_pos']=='DEF']
    GK = team[team['broad_pos']=='GK']
    top11 = team.nlargest(11, 'overall')
    bench = team[~team.index.isin(top11.index)]
    rows.append({
        'team':team_name,
        'atk_overall' : weighted_avg(ATT,'overall','overall'),
        'mid_overall' : weighted_avg(MID,'overall','overall'),
        'def_overall' : weighted_avg(DEF,'overall','overall'),
        'gk_overall' : weighted_avg(GK,'overall','overall'),

        # positional attributes
        'atk_shooting'  : weighted_avg(ATT, 'shooting', 'overall'),
        'atk_pace'      : weighted_avg(ATT, 'pace', 'overall'),
        'atk_dribbling' : weighted_avg(ATT, 'dribbling', 'overall'),
        'atk_xG_per90' : atk_xG,
        'atk_xA_per90' : atk_xA,

        'mid_passing'   : weighted_avg(MID, 'passing', 'overall'),
        'mid_physic'    : weighted_avg(MID, 'physic', 'overall'),

        'def_defending' : weighted_avg(DEF, 'defending', 'overall'),
        'def_physic'    : weighted_avg(DEF, 'physic', 'overall'),

        'top11_overall' : top11['overall'].mean(),
        'squad_depth'   : top11['overall'].mean() - bench['overall'].mean(),
        'avg_caps'      : team['caps'].mean(),

    })

team_vectors = pd.DataFrame(rows)

print(team_vectors[['team', 'atk_xG_per90', 'atk_xA_per90']].dropna().sort_values('atk_xG_per90', ascending=False).head(15))
print(f"\nTeams with xG data: {team_vectors['atk_xG_per90'].notna().sum()} / 48")


                      team  atk_xG_per90  atk_xA_per90
0           Czech Republic      0.696945      0.167570
41                DR Congo      0.668856      0.108361
34                  Norway      0.608116      0.091618
32                  France      0.555792      0.308883
45                 England      0.555448      0.228259
44                 Croatia      0.554986      0.112438
40                Colombia      0.542775      0.369742
18                 Germany      0.530478      0.195325
4   Bosnia and Herzegovina      0.504319      0.091879
36                 Algeria      0.473652      0.219316
1                   Mexico      0.455944      0.071891
30                   Spain      0.432339      0.230264
22                  Sweden      0.392236      0.110876
20                   Japan      0.383300      0.111262
37               Argentina      0.381653      0.238872

Teams with xG data: 39 / 48


In [ ]:
for team_name in ['Czech Republic', 'Colombia', 'DR Congo']:
    team = players[players['team'] == team_name]
    att = team[team['broad_pos'] == 'ATT']
    print(f"\n{team_name} attackers:")
    for _, p in att.iterrows():
        lookup = p['name_lookup']
        if lookup in und_lookup:
            d = und_lookup[lookup]
            print(f"  {p['name']} — {d['time']} mins, xG/90: {d['xG_per90']:.3f}, xA/90: {d['xA_per90']:.3f}")
        else:
            print(f"  {p['name']} — no understat data")


Czech Republic attackers:
  Adam Hložek — no understat data
  Patrik Schick — 2005 mins, xG/90: 0.872, xA/90: 0.143
  Jan Kuchta — no understat data
  Mojmír Chytil — no understat data
  Pavel Šulc — 1562 mins, xG/90: 0.473, xA/90: 0.199
  Tomáš Chorý — no understat data
  Denis Višinský — no understat data

Colombia attackers:
  Luis Díaz — 2464 mins, xG/90: 0.543, xA/90: 0.370
  Jhon Córdoba — no understat data
  Cucho Hernández — no understat data
  Jaminton Campaz — no understat data
  Luis Suárez — no understat data
  Andrés Gómez — no understat data

DR Congo attackers:
  Brian Cipenga — no understat data
  Gaël Kakuta — no understat data
  Meschak Elia — no understat data
  Cédric Bakambu — 542 mins, xG/90: 0.738, xA/90: 0.179
  Fiston Mayele — no understat data
  Yoane Wissa — 492 mins, xG/90: 0.592, xA/90: 0.031
  Simon Banza — no understat data


In [ ]:
fc26 = pd.read_csv("../data/raw/kaggle/FC_26.csv", nrows=5)
print(fc26.columns.tolist())
print(fc26.head(3).to_string())

['player_id', 'player_url', 'fifa_version', 'fifa_update', 'fifa_update_date', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur', 'age', 'dob', 'height_cm', 'weight_kg', 'league_id', 'league_name', 'league_level', 'club_team_id', 'club_name', 'club_position', 'club_jersey_number', 'club_loaned_from', 'club_joined_date', 'club_contract_valid_until_year', 'nationality_id', 'nationality_name', 'nation_team_id', 'nation_position', 'nation_jersey_number', 'preferred_foot', 'weak_foot', 'skill_moves', 'international_reputation', 'work_rate', 'body_type', 'real_face', 'release_clause_eur', 'player_tags', 'player_traits', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control', 'movement_acceleration', 'movement_sprint_sp

In [ ]:
import pandas as pd

fc26 = pd.read_csv("../data/raw/kaggle/FC_26.csv")
print(fc26.columns.tolist())
print(fc26[['long_name', 'short_name', 'dob', 'nationality_name', 'club_name', 'overall']].head(5).to_string())

# statsbomb - check what player info looks like
import json, os
# grab one lineup file
sb_path = "../data/raw/statsbomb/data/lineups"
first_file = os.listdir(sb_path)[0]
with open(f"{sb_path}/{first_file}") as f:
    lineup = json.load(f)
print("\nStatsBomb player fields:")
print(lineup[0]['lineup'][0].keys())
print(lineup[0]['lineup'][0])

['player_id', 'player_url', 'fifa_version', 'fifa_update', 'fifa_update_date', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur', 'age', 'dob', 'height_cm', 'weight_kg', 'league_id', 'league_name', 'league_level', 'club_team_id', 'club_name', 'club_position', 'club_jersey_number', 'club_loaned_from', 'club_joined_date', 'club_contract_valid_until_year', 'nationality_id', 'nationality_name', 'nation_team_id', 'nation_position', 'nation_jersey_number', 'preferred_foot', 'weak_foot', 'skill_moves', 'international_reputation', 'work_rate', 'body_type', 'real_face', 'release_clause_eur', 'player_tags', 'player_traits', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control', 'movement_acceleration', 'movement_sprint_sp

C:\Users\Aaron Varghese\AppData\Local\Temp\ipykernel_23536\691795841.py:3: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  fc26 = pd.read_csv("../data/raw/kaggle/FC_26.csv")


In [ ]:
fc26 = pd.read_csv("../data/raw/kaggle/FC_26.csv", low_memory=False)
squads = pd.read_csv("../data/raw/squads.csv")

# exact DOB match
dob_matches = squads.merge(fc26[['long_name', 'short_name', 'dob', 'nationality_name', 'club_name', 'overall']], 
                            on='dob', 
                            how='left')

matched = dob_matches[dob_matches['overall'].notna()]
print(f"Squad players with DOB match in FC26: {matched['name'].nunique()} / {len(squads)}")

# how many have multiple FC26 players with same DOB (ambiguous)
dob_counts = fc26['dob'].value_counts()
ambiguous = dob_counts[dob_counts > 1]
print(f"DOBs with multiple FC26 players (ambiguous): {len(ambiguous)}")
print(f"Max players sharing a DOB: {ambiguous.max()}")

# match on DOB + nationality
squads['dob'] = pd.to_datetime(squads['dob']).dt.strftime('%Y-%m-%d')
fc26['dob'] = pd.to_datetime(fc26['dob']).dt.strftime('%Y-%m-%d')
nation_map = {
    'Cape Verde'     : 'Cape Verde Islands',
    'Curaçao'        : 'Curacao',
    'Czech Republic' : 'Czechia',
    'DR Congo'       : 'DR Congo',
    'Ivory Coast'    : "Côte d'Ivoire",
    'South Korea'    : 'Korea Republic',
    'Turkey'         : 'Türkiye',
}

squads['nationality_name'] = squads['team'].map(nation_map).fillna(squads['team'])
merged = squads.merge(
    fc26[['long_name', 'short_name', 'dob', 'nationality_name', 'club_name', 'overall']],
    on=['dob', 'nationality_name'],
    how='left'
)

match_counts = merged.groupby('name')['long_name'].count()
unique_matches = match_counts[match_counts == 1]
ambiguous_matches = match_counts[match_counts > 1]
no_match = squads[~squads['name'].isin(merged['name'].dropna())]

print(f"Unique DOB + nationality matches: {len(unique_matches)} / {len(squads)}")
print(f"Ambiguous (multiple FC26 players same DOB + nation): {len(ambiguous_matches)}")
print(f"No match: {len(no_match)}")

print("\nAmbiguous cases:")
print(merged[merged['name'].isin(ambiguous_matches.index)][['name', 'long_name', 'nationality_name', 'club_name', 'overall']].to_string())
print("\nNo match cases (first 20):")
print(no_match[['name', 'team', 'dob', 'club']].head(20).to_string())

Squad players with DOB match in FC26: 1203 / 1248
DOBs with multiple FC26 players (ambiguous): 4499
Max players sharing a DOB: 199
Unique DOB + nationality matches: 823 / 1248
Ambiguous (multiple FC26 players same DOB + nation): 48
No match: 0

Ambiguous cases:
                        name                                                   long_name nationality_name                 club_name  overall
85             Paik Seung-ho                                            Seung-ho Paik백승호   Korea Republic           Birmingham City     72.0
86             Paik Seung-ho                                         Chan-hee Han한찬희 韩赞熙   Korea Republic                  Suwon FC     65.0
96              Oh Hyeon-gyu                                             Hyeon-gyu Oh오현규   Korea Republic                  KRC Genk     71.0
97              Oh Hyeon-gyu                                                  Yool Heo허율   Korea Republic               Ulsan HD FC     63.0
184             Gregor Kobel     

In [ ]:
# get the ambiguous cases
ambiguous_df = merged[merged['name'].isin(ambiguous_matches.index)][
    ['name', 'team', 'club', 'long_name', 'nationality_name', 'club_name', 'overall']
].copy()

# normalize club names for comparison
def norm(s):
    s = str(s).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

ambiguous_df['club_norm'] = ambiguous_df['club'].apply(norm)
ambiguous_df['fc26_club_norm'] = ambiguous_df['club_name'].apply(norm)

# find exact club matches within ambiguous
from rapidfuzz import fuzz
ambiguous_df['club_score'] = ambiguous_df.apply(
    lambda r: fuzz.token_sort_ratio(r['club_norm'], r['fc26_club_norm']), axis=1
)

# for each squad player, take the FC26 row with highest club match score
resolved = ambiguous_df.sort_values('club_score', ascending=False).groupby('name').first().reset_index()

print(f"Resolved via club tiebreaker: {len(resolved)}")
print(f"Top club scores:")
print(resolved[['name', 'club', 'long_name', 'club_name', 'club_score', 'overall']].sort_values('club_score').head(20).to_string())

Resolved via club tiebreaker: 48
Top club scores:
                      name                 club                                                   long_name             club_name  club_score  overall
0       Alejandro Zendejas              América                                            John Jack Skahan  San Jose Earthquakes   22.222222     62.0
34            Oh Hyeon-gyu             Beşiktaş                                             Hyeon-gyu Oh오현규              KRC Genk   25.000000     71.0
7             Cammy Devlin  Heart of Midlothian                                        Cameron Peter Devlin                Hearts   40.000000     71.0
25              Marc Guéhi      Manchester City                           Addji Keaninkin Marc-Israel Guéhi        Crystal Palace   41.379310     82.0
36          Quinten Timber            Marseille                                 Jurriën David Norman Timber               Arsenal   62.500000     82.0
9          Denzel Dumfries          Inter Mi

In [ ]:
print(squads.columns.tolist())
print(squads.head(3).to_string())

['team', 'no', 'position', 'name', 'dob', 'caps', 'goals', 'club']
             team  no position         name         dob  caps  goals           club
0  Czech Republic   1       GK  Matěj Kovář  2000-05-17    20      0  PSV Eindhoven
1  Czech Republic   2       DF   David Zima  2000-11-08    25      1  Slavia Prague
2  Czech Republic   3       DF  Tomáš Holeš  1993-03-31    41      2  Slavia Prague


In [ ]:
squad_nations = set(squads['team'].unique())
fc26_nations = set(fc26['nationality_name'].unique())

in_both = squad_nations & fc26_nations
only_in_squads = squad_nations - fc26_nations

print(f"Teams matching directly: {len(in_both)}")
print(f"\nTeams NOT matching FC26 nationality_name:")
for t in sorted(only_in_squads):
    print(f"  {t}")

Teams matching directly: 41

Teams NOT matching FC26 nationality_name:
  Cape Verde
  Curaçao
  Czech Republic
  DR Congo
  Ivory Coast
  South Korea
  Turkey


In [ ]:
map_df = pd.read_csv("../data/processed/player_id_map.csv")
dupes = map_df[map_df.duplicated(subset=['name', 'team'], keep=False)]
print(f"Duplicate rows: {len(dupes)}")
print(dupes[['name', 'team', 'fc26_long_name', 'fc26_match_method']].to_string())

Duplicate rows: 8
                  name       team                   fc26_long_name fc26_match_method
820  Emiliano Martínez    Uruguay                              NaN   dob_nationality
821  Emiliano Martínez    Uruguay                              NaN   dob_nationality
822  Emiliano Martínez    Uruguay  Damián Emiliano Martínez Romero   dob_nationality
823  Emiliano Martínez    Uruguay  Damián Emiliano Martínez Romero   dob_nationality
987  Emiliano Martínez  Argentina                              NaN   dob_nationality
988  Emiliano Martínez  Argentina                              NaN   dob_nationality
989  Emiliano Martínez  Argentina  Damián Emiliano Martínez Romero   dob_nationality
990  Emiliano Martínez  Argentina  Damián Emiliano Martínez Romero   dob_nationality


In [ ]:
checks = ['Vinícius Júnior', 'Quinten Timber', 'Jonathan David', 'Kenan Yıldız', 'Marc Guéhi']
print(map_df[map_df['name'].isin(checks)][['name', 'team', 'fc26_long_name', 'fc26_overall', 'fc26_match_method', 'understat_name']].to_string())

                 name         team                           fc26_long_name  fc26_overall fc26_match_method            understat_name
139    Jonathan David       Canada                 Jonathan Christian David          82.0   dob_nationality  Jonathan Christian David
214   Vinícius Júnior       Brazil  Vinicius José Paixão de Oliveira Junior          89.0   dob_nationality           Vinícius Júnior
374      Kenan Yıldız       Turkey                             Kenan Yıldız          79.0   dob_nationality              Kenan Yildiz
571    Quinten Timber  Netherlands             Quinten Ryan Crispito Timber          80.0     llm_ambiguous            Quinten Timber
1181       Marc Guéhi      England        Addji Keaninkin Marc-Israel Guéhi          82.0     llm_ambiguous                       NaN


In [ ]:
# fix duplicates
map_df = map_df.sort_values('fc26_long_name', na_position='last')
map_df = map_df.drop_duplicates(subset=['name', 'team'], keep='first')

print(f"Rows after dedup: {len(map_df)}")

# verify
dupes = map_df[map_df.duplicated(subset=['name', 'team'], keep=False)]
print(f"Remaining duplicates: {len(dupes)}")

# save back
map_df.to_csv("../data/processed/player_id_map.csv", index=False)
print("Saved.")

Rows after dedup: 1248
Remaining duplicates: 0
Saved.


In [ ]:
import pandas as pd
import numpy as np

map_df = pd.read_csv("../data/processed/player_id_map.csv")
fc26 = pd.read_csv("../data/raw/kaggle/FC_26.csv", low_memory=False)

# get all FC26 attributes we need — join on long_name
fc26_attrs = fc26[[
    'long_name', 'overall', 'pace', 'shooting', 'passing',
    'dribbling', 'defending', 'physic', 'player_positions'
]].drop_duplicates(subset='long_name', keep='first')

# join map to fc26 attrs
players_master = map_df.merge(
    fc26_attrs,
    left_on='fc26_long_name',
    right_on='long_name',
    how='left',
    suffixes=('', '_fc26')
)

# drop redundant long_name col from fc26
players_master = players_master.drop(columns=['long_name'])

# broad_pos from position column
pos_map = {
    'GK' : 'GK',
    'DF' : 'DEF',
    'MF' : 'MID',
    'FW' : 'ATT',
}
players_master['broad_pos'] = players_master['position'].map(pos_map)

# positional fill for players not in FC26
fill_map = {'ATT': 76, 'MID': 76, 'DEF': 75, 'GK': 74}
for col in ['overall', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']:
    players_master[col] = players_master.apply(
        lambda r: fill_map.get(r['broad_pos'], 75) if pd.isna(r[col]) else r[col],
        axis=1
    )

print(f"Total players: {len(players_master)}")
print(f"FC26 matched: {players_master['fc26_long_name'].notna().sum()}")
print(f"Positional fill: {players_master['fc26_long_name'].isna().sum()}")
print(f"\nBroad pos counts:\n{players_master['broad_pos'].value_counts()}")
print(f"\nNull check:\n{players_master[['overall','pace','shooting','passing','dribbling','defending','physic']].isnull().sum()}")

# save
players_master.to_csv("../data/processed/players_master.csv", index=False)
print("\n✓ players_master.csv saved")

Total players: 1248
FC26 matched: 1045
Positional fill: 203

Broad pos counts:
broad_pos
DEF    422
MID    369
ATT    312
GK     145
Name: count, dtype: int64

Null check:
overall      0
pace         0
shooting     0
passing      0
dribbling    0
defending    0
physic       0
dtype: int64

✓ players_master.csv saved


In [9]:
results = pd.read_csv("../data/raw/kaggle/shootouts.csv")
results.tail(30)

,date,home_team,away_team,winner,first_shooter
648,2025-06-08,Portugal,Spain,Portugal,Portugal
649,2025-06-28,Panama,Honduras,Honduras,Panama
650,2025-06-29,Canada,Guatemala,Guatemala,Canada
651,2025-06-29,United States,Costa Rica,United States,Costa Rica
652,2025-07-13,Gozo,Shetland,Gozo,Gozo
653,2025-07-17,Gozo,Western Isles,Gozo,NaN
654,2025-08-13,United States Virgin Islands,Turks and Caicos Islands,United States Virgin Islands,NaN
655,2025-11-13,Iran,Cape Verde,Iran,NaN
656,2025-11-16,Nigeria,DR Congo,DR Congo,Nigeria
657,2025-11-17,Cape Verde,Egypt,Egypt,NaN
